In [29]:
import requests
import tarfile
import io
from bs4 import BeautifulSoup
from langchain_core.documents import Document
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
from dotenv import load_dotenv
load_dotenv()
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from agent.agent_modules import VectorSearch
vs = VectorSearch(region = "europe-north2", model_name="text-embedding-004")


In [30]:
def fetch_json(endpoint : str):
    base = "https://api.lovdata.no/" + endpoint
    try:
        response = requests.get(base,)
        if response.status_code == 200:
            return response.json()
    except Exception as e:
        logger.error(f"Error fetching JSON from {base}: {e}")
        return None

def fetch_zip(endpoint : str) -> tarfile.TarFile:
    base = "https://api.lovdata.no/" + endpoint
    try:
        response = requests.get(base,)
        if response.status_code == 200:
            tar_file = tarfile.open(fileobj=io.BytesIO(response.content), mode='r:bz2')
            return tar_file
    except Exception as e:
        logger.error(f"Error fetching zip from {base}: {e}")
        return None

def normalize_metadata(metadata):
    """Translate and normalize to English snake_case"""
    mapping = {
        'Datokode': 'date_code',
        'DokumentID': 'document_id', 
        'Departement': 'department',
        'I kraft frå': 'in_force_from',
        'I kraft fra': 'in_force_from',
        'Ikrafttreding av siste endring': 'last_amendment_effective',
        "Ikrafttredelse av siste endring": 'last_amendment_effective',
        'Sist endra ved': 'last_amended_by',
        'Sist endret ved': 'last_amended_by',
        'Rettsområde': 'legal_area',
        'Korttittel': 'short_title',
        "Annet om dokumentet": 'other_about_document',
        'Tittel': 'title',
        'RefID': 'ref_id',
        "Innhold": "contents",
    }
    return {mapping.get(k, k.lower().replace(' ', '_')): v for k, v in metadata.items()}

def extract_info(xml_string) -> tuple[list[dict], dict]:
    """Extract structured paragraphs and metadata from Lovdata XML/HTML
    
    Args:
        xml_string (str): XML/HTML string from Lovdata
        
    Returns:
        tuple[list[dict], dict]: List of paragraph dicts with hierarchy and metadata
    """
    soup = BeautifulSoup(xml_string, 'html.parser')
    
    # Ekstraher metadata
    metadata = {}
    dl = soup.find('dl', class_='data-document-key-info')
    if dl:
        for dt, dd in zip(dl.find_all('dt'), dl.find_all('dd')):
            key = dt.get_text(strip=True)
            value = dd.get_text(strip=True)
            metadata[key] = value
    
    # Ekstraher strukturerte paragrafer
    paragraphs = []
    main = soup.find('main', class_='documentBody')
    
    if main:
        # Hent alle <article class="legalArticle"> (faktiske lovparagrafer)
        for article in main.find_all('article', class_='legalArticle'):
            # Hent paragrafnummer
            header = article.find('h4', class_='legalArticleHeader')
            para_num = header.get_text(strip=True) if header else ''
            
            # Hent lovtekst (alle <article class="legalP"> inni)
            legal_texts = article.find_all('article', class_='legalP')
            text = '\n'.join([p.get_text(strip=True) for p in legal_texts])
            
            paragraphs.append({
                'paragraph_number': para_num,
                'text': text,
                'data_name': article.get('data-name', ''),
                'id': article.get('id', '')
            })
    
    return paragraphs, metadata

def mk_doc(file,data) -> Document:
        try:
            xml = data.extractfile(file).read().decode('utf-8')
            paragraphs, law_metadata = extract_info(xml)
            law_metadata_norm = normalize_metadata(law_metadata)
            
            for para in paragraphs:
                # Kombiner lov-metadata med paragraf-metadata (IKKE mutér para)
                doc_meta = {
                    **law_metadata_norm,
                    'paragraph_number': para['paragraph_number'],
                    'data_name': para['data_name'],
                    'paragraph_id': para['id']  # Renamed for clarity
                }
                logger.info(f"Extracted {law_metadata_norm.get('title')}")
            return Document(
                page_content=para['text'], 
                metadata=doc_meta
            )
        except Exception as e:
            logger.error(f"Error processing {file.name}: {e}")


In [2]:
vector_store = vs.init_vector_store(table_name="laws")

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"
INFO:langchain_google_community.bq_storage_vectorstores._base:BigQuery table master-thesis-26.vector_store.laws initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=master-thesis-26&ws=!1m5!1m4!4m3!1smaster-thesis-26!2svector_store!3slaws


## Load laws to BQ

In [31]:
public = fetch_json("v1/publicData/list")
filename = "gjeldende-lover.tar.bz2"
data = fetch_zip(f"/v1/publicData/get/{filename}")
public

[{'filename': 'lovtidend-avd1-2026.tar.bz2',
  'description': 'Norsk Lovtidend avd. I - lover og sentrale forskrifter',
  'sizeBytes': '14674',
  'lastModified': '2026-01-13T02:30:00Z'},
 {'filename': 'gjeldende-sentrale-forskrifter.tar.bz2',
  'description': 'Gjeldende sentrale forskrifter, ajourført med endringer',
  'sizeBytes': '20513205',
  'lastModified': '2026-01-13T02:31:00Z'},
 {'filename': 'gjeldende-lover.tar.bz2',
  'description': 'Gjeldende lover, ajourført med endringer',
  'sizeBytes': '5813718',
  'lastModified': '2026-01-13T02:31:00Z'},
 {'filename': 'lovtidend-avd1-2001-2025.tar.bz2',
  'description': 'Norsk Lovtidend avd. I - lover og sentrale forskrifter',
  'sizeBytes': '69179404',
  'lastModified': '2026-01-13T02:30:00Z'}]

In [ ]:
public = fetch_json("v1/publicData/list")
filename = "gjeldende-lover.tar.bz2"
data = fetch_zip(f"/v1/publicData/get/{filename}")

laws = []
for idx, file in enumerate(data.getmembers()):
            laws.append(mk_doc(file,data))
# Batch upload
#vector_store.add_documents(laws)

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"
INFO:langchain_google_community.bq_storage_vectorstores._base:BigQuery table master-thesis-26.vector_store.laws initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=master-thesis-26&ws=!1m5!1m4!4m3!1smaster-thesis-26!2svector_store!3slaws


## Test BQ vector Store for laws

In [ ]:
res = vs.query(query = "Jeg har en boligkjøpstvist (bruktbolig) hvor jeg vil heve kjøpet. Hvem lov er aktuell? Det er solgt med uaktsomhet og store feil og mangler",
               table_name="laws")

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"
INFO:langchain_google_community.bq_storage_vectorstores._base:BigQuery table master-thesis-26.vector_store.laws initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=master-thesis-26&ws=!1m5!1m4!4m3!1smaster-thesis-26!2svector_store!3slaws
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:batchEmbedContents "HTTP/1.1 200 OK"


In [35]:
for i in res:
    c = json.loads(i)
    print(c.get("metadata").get("title"))
    print("\t")
    print(c.get("page_content"))
    print("\n\n")

Lov om eierseksjoner (eierseksjonsloven)
	
Med leie til eie menes i denne loven avtale mellom en juridisk person og en eller flere fysiske personer om leie av bolig, der den eller de fysiske personene innenfor en tidsbestemt periode har rett til å kjøpe boligen.
Med deleie menes i denne loven avtale om delt eierskap av en bolig mellom en juridisk person og en eller flere fysiske personer, der den eller de fysiske personene har rett til å kjøpe hele boligen. Den eller de fysiske personene må eie minst 50 prosent av boligen og ha en eksklusiv bruksrett til hele boligen.
Departementet kan gi forskrift om unntak fra begrensningen i adgangen til å kjøpe eller på annen måte erverve boligseksjoner etter§ 23, slik at inntil 50 prosent av boligseksjonene i sameiet kan erverves av juridiske personer som tilbyr en bolig med avtale om leie til eie eller deleie. Forskriften skal stille nærmere vilkår til avtalene om leie til eie og deleie.



Lov om eksplosive varer [gjelder bare for Svalbard]
	
Ti

In [15]:
res

['{"id":null,"metadata":{"doc_id":"3d1cfc3ad89e42aa9d416231b8e1c19f","date_code":"LOV-2017-06-16-65","document_id":"NL/lov/2017-06-16-65","department":"Kommunal- og distriktsdepartementet","last_amendment_effective":"2026-01-01","last_amended_by":"lov/2025-12-19-114fra 2026-01-01","legal_area":"Anskaffelser, avtaler, bygg og entrepriser>BoligoppføringFast eiendoms rettsforhold>SameieFast eiendoms rettsforhold>Tinglysing. RegistreringKonkurs, gjeld og pant>PantTvangsfullbyrdelse>Tvangsgrunnlag og tvangskraft","rettet":"2022-03-23 (tegnsetting i lister tilpasset universell utforming)","short_title":"Eierseksjonsloven – eiersl","title":"Lov om eierseksjoner (eierseksjonsloven)","other_about_document":"Loven trådte i kraft 1 jan 2018, med unntak for§ 9, som trådte i kraft 1 juli 2018.Jf.tidligerelover 4 mars 1983 nr. 7og23 mai 1997 nr. 31.","ref_id":"lov/2017-06-16-65","contents":"Lov om eierseksjoner (eierseksjonsloven)Kapittel I. Innledende bestemmelser§ 1. Lovens formål§ 2. Lovens virke